In [1]:
from pathlib import Path

# The notebooks are stored in <package>/notebooks.
notebooks_dir = Path.cwd()
if notebooks_dir.name != "notebooks":
    candidate = notebooks_dir / "notebooks"
    if candidate.exists():
        notebooks_dir = candidate

project_root = notebooks_dir.parent
nhanes_dir = project_root / "src" / "nhanes"
jordan_dir = project_root / "src" / "jordan_external_validation"
expected_nhanes = project_root / "outputs" / "nhanes"
expected_jordan = project_root / "outputs" / "jordan"

assert (project_root / "README.md").exists(), project_root
print("Project root:", project_root)

Project root: <project_root>


In [2]:
import subprocess
import sys

code_dir = jordan_dir / "external_validation"
participant_file = jordan_dir / "raw" / "jor2019.dta"
results_dir = code_dir / "jordan_results"

RUN_JORDAN_VALIDATION = False  # Set True for the complete 2,000-resample validation.

command = [
    sys.executable, "-u", str(code_dir / "run_jordan_external_validation.py"),
    str(participant_file), "--output", str(results_dir), "--replicates", "2000"
]
print("Command:", " ".join(command))

if RUN_JORDAN_VALIDATION:
    assert participant_file.exists(), "Licensed Jordan DTA file is missing"
    assert (jordan_dir / "raw" / "jor2019.csv").exists(), "Licensed Jordan CSV file is missing"
    subprocess.run(command, cwd=jordan_dir, check=True)
    subprocess.run([sys.executable, str(code_dir / "verify_jordan_external_validation.py")], cwd=jordan_dir, check=True)
    print("PASS: frozen Jordan validation completed and verified.")
else:
    print("SKIPPED: set RUN_JORDAN_VALIDATION = True for an authorized rerun.")

Command: python -u <project_root>\src\jordan_external_validation\external_validation\run_jordan_external_validation.py <project_root>\src\jordan_external_validation\raw\jor2019.dta --output <project_root>\src\jordan_external_validation\external_validation\jordan_results --replicates 2000
SKIPPED: set RUN_JORDAN_VALIDATION = True for an authorized rerun.


In [3]:
import json
import pandas as pd

live = results_dir
results = live if (live / "jordan_external_validation_headline_results.csv").exists() else expected_jordan
print("Reading aggregate results from:", results)

headline = pd.read_csv(results / "jordan_external_validation_headline_results.csv")
contrasts = pd.read_csv(results / "jordan_external_validation_auc_contrasts.csv")
display(headline.round(4))
display(contrasts.round(4))

Reading aggregate results from: <project_root>\outputs\jordan


,configuration,n,cases,weighted_auc,auc_lower_psu,auc_upper_psu,brier,calibration_slope,calibration_in_the_large,sensitivity_at_0_20,specificity_at_0_20
0,BASE,885,249,0.7694,0.7159,0.8174,0.1725,0.6541,1.3032,0.3589,0.8976
1,BMI,885,249,0.7617,0.7096,0.8109,0.1797,0.6141,1.5387,0.3024,0.9204
2,WHR,885,249,0.7378,0.6858,0.7872,0.1888,0.4248,1.8005,0.2562,0.9089
3,BOTH,885,249,0.7391,0.6876,0.7880,0.1898,0.4253,1.8728,0.2593,0.9213


,contrast,auc_difference,lower_psu,upper_psu
0,BMI minus WHR,0.0239,-0.0018,0.0559
1,WHR minus BMI,-0.0239,-0.0559,0.0018
2,BMI plus WHR minus BMI,-0.0226,-0.0529,0.0021


In [4]:
bmi_auc = headline.loc[headline.configuration.eq("BMI"), "weighted_auc"].iloc[0]
whr_auc = headline.loc[headline.configuration.eq("WHR"), "weighted_auc"].iloc[0]

assert round(bmi_auc, 4) == 0.7617
assert round(whr_auc, 4) == 0.7378
print("PASS: external headline values match the expected results.")

PASS: external headline values match the expected results.
